In [1]:
# Cell 1 - System setup and clone fish-speech repo
import os

print("🧹 1. Cleaning old environment...")
%cd /content/
!rm -rf /content/fish-speech

print("📦 2. Installing system audio dependencies FIRST...")
!apt-get update -qq && apt-get install -y portaudio19-dev ffmpeg -qq

print("⬇️ 3. Cloning repository and rolling back to stable commit...")
!git clone https://github.com/fishaudio/fish-speech.git /content/fish-speech
%cd /content/fish-speech
# Checking out your exact requested commit that supports tiktoken + S1-Mini
!git checkout d3df50503b36314a964f66cac1af1e19e95bcfa3

print("🔧 4. Installing Python libraries...")
# Install PyTorch with CUDA support first to avoid version conflicts
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# Install the repo in editable mode so internal paths work
!pip install -q -e .
!pip install -q gradio pydub soundfile pyrootutils loguru hydra-core

print("\n✅ CORE SETUP COMPLETE!")
print("🛑 CRITICAL: Go to 'Runtime' -> 'Restart session' NOW before running the next cell!")

🧹 1. Cleaning old environment...
/content
📦 2. Installing system audio dependencies FIRST...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
⬇️ 3. Cloning repository and rolling back to stable commit...
Cloning into '/content/fish-speech'...
remote: Enumerating objects: 6594, done.
remote: Counting objects: 100% (1080/1080), done.
remote: Compressing objects: 100% (295/295), done.
remote: Total 6594 (delta 899), reused 785 (delta 785), pack-reused 5514 (from 1)
Receiving objects: 100% (6594/6594), 28.17 MiB | 13.71 MiB/s, done.
Resolving deltas: 100% (4325/4325), done.
/content/fish-speech
Note: switching to 'd3df50503b36314a964f66cac1af1e19e95bcfa3'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching b

In [3]:
# Cell 2 - Download Model and Reference Audio
from huggingface_hub import snapshot_download, hf_hub_download
import soundfile as sf
import os

%cd /content/fish-speech/

print("⬇️ 1. Downloading S1-Mini Model...")
model_dir = snapshot_download(
    repo_id="Vansh45673/fishspeechmine",
    local_dir="/content/fish-speech/checkpoints/openaudio-s1-mini",
    local_dir_use_symlinks=False
)
print(f"✓ Model downloaded to: {model_dir}")

print("\n⬇️ 2. Downloading Reference Audio...")
os.makedirs("/content/fish-speech/references", exist_ok=True)
ref_path = hf_hub_download(
    repo_id="Vansh45673/home",
    filename="voice cloning audio.wav", # Make sure this matches your HF repo file!
    local_dir="/content/fish-speech/references"
)

# Convert to absolute path to prevent subprocess issues
ref_path = os.path.abspath(ref_path)
print(f"✓ Reference voice: {ref_path}")

# Check reference audio duration
ref_audio, ref_sr = sf.read(ref_path)
print(f"📊 Reference: {len(ref_audio) / ref_sr:.1f}s @ {ref_sr}Hz")

/content/fish-speech
⬇️ 1. Downloading S1-Mini Model...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

✓ Model downloaded to: /content/fish-speech/checkpoints/openaudio-s1-mini

⬇️ 2. Downloading Reference Audio...


voice cloning audio.wav:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

✓ Reference voice: /content/fish-speech/references/voice cloning audio.wav
📊 Reference: 10.1s @ 48000Hz


In [4]:
# Cell 3 - FIXED Inference Wrapper for d3df505
import os
import subprocess
import sys
import shutil

def generate_audio_chunk(
    text: str,
    reference_audio: str,
    output_path: str,
    reference_text: str = "",
    top_p: float = 0.7,
    temperature: float = 0.7
):
    repo_root = "/content/fish-speech"
    absolute_codec_path = os.path.join(repo_root, "checkpoints/openaudio-s1-mini/codec.pth")
    absolute_model_path = os.path.join(repo_root, "checkpoints/openaudio-s1-mini")

    my_env = os.environ.copy()
    my_env["PYTHONPATH"] = repo_root

    # STEP 1: Encode Reference Audio
    print("🎤 Encoding reference audio...")
    cmd_encode = [
        sys.executable, "fish_speech/models/dac/inference.py",
        "-i", reference_audio,
        "--checkpoint-path", absolute_codec_path
    ]
    result = subprocess.run(cmd_encode, capture_output=True, text=True, cwd=repo_root, env=my_env)
    if result.returncode != 0:
        raise RuntimeError(f"Encoding failed: {result.stderr}")

    ref_npy = os.path.join(repo_root, "fake.npy")
    if not os.path.exists(ref_npy):
        raise RuntimeError("Reference encoding failed - fake.npy not found")

    # STEP 2: Generate Semantic Tokens
    print("📝 Generating semantic tokens...")
    cmd_gen = [
        sys.executable, "fish_speech/models/text2semantic/inference.py",
        "--text", text,
        "--prompt-tokens", ref_npy,
        "--checkpoint-path", absolute_model_path, # The previous AI completely missed this flag!
        "--num-samples", "1",
        "--top-p", str(top_p),
        "--temperature", str(temperature)
    ]
    if reference_text:
        cmd_gen.extend(["--prompt-text", reference_text])

    result = subprocess.run(cmd_gen, capture_output=True, text=True, cwd=repo_root, env=my_env)
    if result.returncode != 0:
        print(f"STDOUT: {result.stdout}\nSTDERR: {result.stderr}")
        raise RuntimeError(f"Semantic generation failed: {result.stderr}")

    # Handle output paths (sometimes older versions output to temp/)
    codes_file = os.path.join(repo_root, "codes_0.npy")
    if not os.path.exists(codes_file):
        temp_codes = os.path.join(repo_root, "temp", "codes_0.npy")
        if os.path.exists(temp_codes):
            shutil.move(temp_codes, codes_file)
        else:
            raise RuntimeError("Semantic token generation failed - codes_0.npy not found")

    # STEP 3: Decode to Audio
    print("🔊 Decoding to audio...")
    cmd_decode = [
        sys.executable, "fish_speech/models/dac/inference.py",
        "-i", codes_file,
        "--checkpoint-path", absolute_codec_path
    ]
    result = subprocess.run(cmd_decode, capture_output=True, text=True, cwd=repo_root, env=my_env)
    if result.returncode != 0:
        raise RuntimeError(f"Decoding failed: {result.stderr}")

    fake_wav = os.path.join(repo_root, "fake.wav")
    if not os.path.exists(fake_wav):
        raise RuntimeError("Decoding failed - fake.wav not found")

    # Move to final chunk destination
    shutil.move(fake_wav, output_path)

    # Cleanup temp files so they don't bleed into the next chunk
    for f in [ref_npy, codes_file]:
        if os.path.exists(f):
            os.remove(f)

    return output_path

print("✅ Inference wrapper ready!")

✅ Inference wrapper ready!


In [5]:
# Cell 4 - PRODUCTION VERSION: FFmpeg Concatenation (ZERO RAM)
import numpy as np
import soundfile as sf
import os
import re
import gc
import torch
import subprocess
import tempfile

# Add your specific transcription reference here
REFERENCE_TEXT = "In November 2025, Google released Gemini 3, its most powerful language model yet. And it was so much better than GPT-5 that in response, OpenAI declared a code red. That's what's been making most of the headlines lately. But here's the thing: Gemini 3 isn't just a threat to OpenAI.Google is now competing with NVIDIA, Oracle, Microsoft, Meta, AMD—basically every AI company you could name. And they're doing it in a way that no other company possibly could. The more I research this, the more it starts to look like no matter what happens in AI going forward, Google is going to win.But to understand why, you have to go back to when Google's AI strategy was... still a complete mess. Funny thing is, it wasn't even that long ago. Let's go back to 2020, before all of this AI stuff happened. At the time, Google was in the middle of one of the most important trials in its history. The U.S. government had accused them of illegally monopolizing the search advertising market, and the jury actually found them guilty. It even looked like Google would be forced to spin off Chrome and Android into their own separate companies."

def chunk_text_simple(text: str, max_chars: int = 300) -> list:
    chunks = []
    sentences = re.split(r'([.!?]\s+)', text)
    current = ""
    for i in range(0, len(sentences), 2):
        sentence = sentences[i] + (sentences[i+1] if i+1 < len(sentences) else '.')
        if len(current) + len(sentence) <= max_chars:
            current += " " + sentence
        else:
            if current:
                chunks.append(current.strip())
            current = sentence
    if current:
        chunks.append(current.strip())
    return [c for c in chunks if c]

def concat_with_ffmpeg(chunk_files, output_path):
    temp_dir = tempfile.mkdtemp()
    concat_list = os.path.join(temp_dir, "concat_list.txt")

    with open(concat_list, 'w') as f:
        for chunk_file in chunk_files:
            abs_path = os.path.abspath(chunk_file)
            f.write(f"file '{abs_path}'\n")

    print(f"\n💾 FFmpeg: Concatenating {len(chunk_files)} files (disk-based)...")
    cmd = ['ffmpeg', '-f', 'concat', '-safe', '0', '-i', concat_list, '-c', 'copy', '-y', output_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    os.remove(concat_list)
    os.rmdir(temp_dir)

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg concat failed: {result.stderr}")
    print(f"✅ FFmpeg concatenation complete!")

def generate_long_audio_safe(text: str, reference_audio: str, output_path: str, temperature: float = 0.4, top_p: float = 0.6):
    chunks = chunk_text_simple(text, max_chars=300)

    print(f"\n{'='*60}")
    print(f"🎙️ GENERATING {len(chunks)} CHUNKS")
    print(f"📝 Total text: {len(text)} characters")
    print(f"{'='*60}\n")

    chunk_files = []
    temp_dir = "/content/temp_chunks"
    os.makedirs(temp_dir, exist_ok=True)

    for i, chunk in enumerate(chunks):
        print(f"[{i+1}/{len(chunks)}] Processing {len(chunk)} chars...")
        temp_output = os.path.join(temp_dir, f"chunk_{i:03d}.wav")

        try:
            generate_audio_chunk(
                text=chunk,
                reference_audio=reference_audio,
                output_path=temp_output,
                reference_text=REFERENCE_TEXT,
                temperature=temperature,
                top_p=top_p
            )
            if os.path.exists(temp_output):
                chunk_files.append(temp_output)
            else:
                raise FileNotFoundError("Output file not created")
        except Exception as e:
            print(f"    ⚠️ Chunk failed: {e}")
            print(f"    Creating 2s silence as fallback...")
            silence = np.zeros(int(44100 * 2), dtype=np.float32)
            sf.write(temp_output, silence, 44100)
            chunk_files.append(temp_output)

        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*60}")
    concat_with_ffmpeg(chunk_files, output_path)

    for f in chunk_files:
        if os.path.exists(f): os.remove(f)

    if os.path.exists(output_path):
        print(f"\n✅ GENERATION COMPLETE! Saved to: {output_path}")
    return output_path

print("✅ Chunking system ready!")

✅ Chunking system ready!


In [ ]:
# Cell 5 - Gradio Interface
import gradio as gr
import traceback

def process_input(file, text_input, temperature, top_p):
    if 'ref_path' not in globals():
        return None, "❌ **Setup Error**: Run Cell 2 first to load reference audio."

    if text_input and text_input.strip():
        text = text_input.strip()
    elif file:
        try:
            with open(file.name, 'r', encoding='utf-8') as f:
                text = f.read().strip()
        except Exception as e:
            return None, f"❌ Failed to read file: {e}"
    else:
        return None, "❌ Please provide text OR upload a file"

    try:
        output = "/content/final_voice_output.wav"
        generate_long_audio_safe(text=text, reference_audio=ref_path, output_path=output, temperature=temperature, top_p=top_p)
        return output, "✅ **Generation Complete!**"
    except Exception as e:
        return None, f"❌ **Error**:\n```\n{str(e)}\n\n{traceback.format_exc()}\n```"

with gr.Blocks(title="OpenAudio S1-Mini TTS") as demo:
    gr.Markdown("# 🎙️ OpenAudio S1-Mini Voice Cloning (Fixed)")
    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(label="📝 Paste Text", lines=8)
            file_input = gr.File(label="📄 OR Upload .txt file", file_types=[".txt"])
        with gr.Column(scale=1):
            temperature = gr.Slider(minimum=0.4, maximum=1.0, value=0.4, step=0.05, label="🌡️ Temperature")
            top_p = gr.Slider(minimum=0.5, maximum=1.0, value=0.6, step=0.05, label="🎯 Top-p")

    generate_btn = gr.Button("🎙️ Generate Speech", variant="primary", size="lg")
    audio_output = gr.Audio(label="🔊 Generated Audio", type="filepath")
    status_output = gr.Markdown("*Waiting for input...*")

    generate_btn.click(fn=process_input, inputs=[file_input, text_input, temperature, top_p], outputs=[audio_output, status_output])

demo.launch(share=True, debug=True, allowed_paths=["/content"])

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f522231d730484cde.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🎙️ GENERATING 1 CHUNKS
📝 Total text: 238 characters

[1/1] Processing 241 chars...
🎤 Encoding reference audio...
📝 Generating semantic tokens...
🔊 Decoding to audio...
